# Analysis of Gain neural network

In [8]:
from bargain import generate_observation
from bargain.networks import GainNN, get_device

import torch
import numpy as np

device = get_device()

In [9]:
path = "data/models/GainNN-2026-06-28_19-52-14.pth"
model = GainNN()
model.load_state_dict(
    torch.load(path)
)

model.eval()

GainNN(
  (instance): Sequential(
    (0): Linear(in_features=48, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (coalition): Sequential(
    (0): Linear(in_features=3, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (head): Sequential(
    (0): Linear(in_features=512, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [10]:
(instance, assignment,
 coalition, char_function,
 shapley, nucleolus) = generate_observation(n_depots = 3,
                                            n_customers = 9,
                                            radius = 0.6)

instance = instance.reshape(1, 12, 5)
coalition = np.array(coalition).reshape(1, 8, 3)
char_function = np.array(char_function).reshape(1, 8)

In [11]:
instance, coalition, char_function = model.transform(instance,
                                                     coalition,
                                                     char_function,
                                                     device)

In [12]:
with torch.no_grad():
    _values = model(instance.float(), coalition.float())

print('pred: ', _values.t())
print('actual: ', char_function.t())

pred:  tensor([[-1.0934e-03, -1.0934e-03, -1.0933e-03,  2.2951e-02,  5.5686e-01,
          6.0363e-01,  5.9334e-01,  2.0122e+00]])
actual:  tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.6733, 0.6851, 0.7076, 2.2212]],
       dtype=torch.float64)
